# Full Graph-Flashback on Gowalla (Kaggle)
This notebook downloads Gowalla in Kaggle, objectively selects a city, trains train-only STKG/TransE/Graph-Flashback, evaluates Acc@k, MAP@k and MRR, and creates figures.


In [ ]:
from pathlib import Path
PROJECT = Path('/kaggle/working/GeoGNNProject')
%cd $PROJECT


In [ ]:
!python -m pip install -q -r requirements.txt


## Smoke test


In [ ]:
!python scripts/make_synthetic_data.py
!python -m flashback.pipeline --config configs/gowalla_smoke.yaml --stage all
!pytest -q


## Full Gowalla run
Internet and GPU should be enabled. Each stage is separate so Kaggle output can be saved between stages.


In [ ]:
!python -m flashback.pipeline --config configs/gowalla_auto.yaml --stage download


In [ ]:
!python -m flashback.pipeline --config configs/gowalla_auto.yaml --stage prepare
import pandas as pd
display(pd.read_csv('data/processed/city_ranking.csv'))


In [ ]:
!python -m flashback.pipeline --config configs/gowalla_auto.yaml --stage stkg
!python -m flashback.pipeline --config configs/gowalla_auto.yaml --stage kge
!python -m flashback.pipeline --config configs/gowalla_auto.yaml --stage graphs


In [ ]:
!python -m flashback.pipeline --config configs/gowalla_auto.yaml --stage train
!python -m flashback.pipeline --config configs/gowalla_auto.yaml --stage analyze


In [ ]:
import json, glob
for p in glob.glob('artifacts/results/*_metrics.json'):
    print(p); display(json.load(open(p)))


In [ ]:
from IPython.display import display, Image
for p in sorted(Path('artifacts/figures').glob('*.png')):
    print(p.name); display(Image(filename=str(p)))


In [ ]:
!zip -r /kaggle/working/graph_flashback_output.zip artifacts checkpoints data/processed data/kge data/graphs configs
